# E-002 Baseline - Nemotron-3-Nano-30B + LoRA SFT
Mirrors the official NVIDIA demo: RTX Pro 6000 (96GB), kagglehub model download, full bf16 (no 4-bit).

In [ ]:
import os, sys, json, subprocess, glob
import torch
print('CUDA:', torch.cuda.is_available(), '| count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {p.name} | {p.total_memory/1024**3:.1f} GB | sm_{p.major}{p.minor}')
print('=== /kaggle/input ===')
for root, dirs, files in os.walk('/kaggle/input'):
    d = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if d <= 2: print(' ', root, '| files:', files[:5])
    if d >= 2: dirs.clear()

In [ ]:
# Install Mamba deps (Nemotron is a Mamba-hybrid). Internet is enabled.
for pkg in ['causal-conv1d', 'mamba-ssm']:
    mod = pkg.replace('-', '_')
    try:
        __import__(mod); print(f'{pkg}: ok')
    except ImportError:
        print(f'installing {pkg}...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', pkg], capture_output=True, text=True)
        print(f'  rc={r.returncode}', r.stderr[-400:] if r.returncode else '')

In [ ]:
# Model via kagglehub (official demo pattern)
import kagglehub
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print('MODEL_PATH:', MODEL_PATH)
print('files:', os.listdir(MODEL_PATH)[:10])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# 96GB GPU -> full bf16, no quantization needed
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('Model + tokenizer loaded.')

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
LORA_RANK = 32
lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=16,
    target_modules=r'.*\.(in_proj|out_proj|up_proj|down_proj)$',  # demo's proven targeting
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load competition train.csv (auto-detect path)
import csv
cands = glob.glob('/kaggle/input/*/train.csv')
assert cands, 'train.csv not found - check competition is attached'
TRAIN_CSV = cands[0]; print('train.csv:', TRAIN_CSV)
SAMPLE_SIZE = 50
train_data = []
with open(TRAIN_CSV, encoding='utf-8') as f:
    for row in csv.DictReader(f):
        p = row.get('prompt') or row.get('question') or ''
        a = row.get('answer') or row.get('completion') or ''
        if p and a: train_data.append({'prompt': p, 'completion': str(a)})
        if len(train_data) >= SAMPLE_SIZE: break
print(f'Loaded {len(train_data)} examples')

In [ ]:
# 1-epoch LoRA SFT (mask: answer tokens only)
MAX_SEQ, GA = 2048, 8
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
dev = next(model.parameters()).device
def datum(p, a):
    pt = tokenizer(p, add_special_tokens=False)['input_ids']
    at = tokenizer(a, add_special_tokens=False)['input_ids'] + [tokenizer.eos_token_id]
    t = (pt + at)[:MAX_SEQ]; m = ([0]*len(pt) + [1]*len(at))[:MAX_SEQ]
    return torch.tensor(t).unsqueeze(0), torch.tensor(m).unsqueeze(0)
model.train(); step = 0; rl = 0.0
for d in train_data:
    ids, mask = datum(d['prompt'], d['completion']); ids, mask = ids.to(dev), mask.to(dev)
    lg = model(input_ids=ids).logits
    sl, slb, sm = lg[..., :-1, :].contiguous(), ids[..., 1:].contiguous(), mask[..., 1:].contiguous()
    lf = torch.nn.CrossEntropyLoss(reduction='none')
    loss = (lf(sl.view(-1, sl.size(-1)), slb.view(-1)) * sm.view(-1)).sum() / sm.sum().clamp(min=1)
    (loss / GA).backward(); rl += loss.item()
    if (step + 1) % GA == 0: opt.step(); opt.zero_grad()
    step += 1
    if step % 5 == 0: print(f'  step {step}/{len(train_data)} loss={rl/5:.4f}'); rl = 0.0
print('Training done.')

In [ ]:
# Save adapter to /kaggle/working (host reads this)
OUT = '/kaggle/working'
model.save_pretrained(OUT)
cfg = json.load(open(os.path.join(OUT, 'adapter_config.json')))
assert cfg.get('r', 999) <= 32, 'rank>32'
print('adapter saved, r=', cfg.get('r'))
for f in os.listdir(OUT):
    if f.startswith('adapter'): print(' ', f, os.path.getsize(os.path.join(OUT, f)) // 1024, 'KB')

In [ ]:
# Quick self-eval with boxed extraction
def boxed(t):
    if not t: return None
    i = t.rfind(chr(92) + 'boxed{')
    if i < 0: return None
    s = i + 7; d = 1
    for j in range(s, len(t)):
        if t[j] == '{': d += 1
        elif t[j] == '}':
            d -= 1
            if d == 0: return t[s:j]
    return None
def ok(pr, go):
    p = boxed(pr)
    if p is None: return False
    g = boxed(go) or str(go).strip(); p = p.strip()
    if p == g.strip(): return True
    try: return abs(float(p) - float(g)) <= 1e-2
    except: return False
model.eval(); preds = []
with torch.no_grad():
    for i, d in enumerate(train_data[:10]):
        inp = tokenizer(d['prompt'], return_tensors='pt').to(dev)
        o = model.generate(**inp, max_new_tokens=256, do_sample=False)
        txt = tokenizer.decode(o[0], skip_special_tokens=True)[len(d['prompt']):].strip()
        preds.append(ok(txt, d['completion']))
score = sum(preds) / len(preds)
json.dump({'score': score, 'n': len(preds), 'correct': sum(preds)}, open(os.path.join(OUT, 'cv_score.json'), 'w'))
print(f'=== CV {score:.4f} ({sum(preds)}/{len(preds)}) ===')

In [ ]:
# Package submission.zip (adapter files at root)
subprocess.run('cd /kaggle/working && (zip -j submission.zip adapter_config.json adapter_model.safetensors || zip -j submission.zip adapter_config.json adapter_model.bin)', shell=True)
print('submission.zip exists:', os.path.exists('/kaggle/working/submission.zip'))
subprocess.run('cd /kaggle/working && unzip -l submission.zip', shell=True)